In [1]:
import sys
import torch
import torch.nn as nn
import math
from feat.utils import set_torch_device
import torch.nn.functional as F
from copy import deepcopy
import numpy as np
from skimage import draw
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from feat.utils.image_operations import extract_face_from_landmarks

from PIL import Image
from itertools import product
import os 
from torchvision.transforms import Compose, Normalize, Grayscale

import pandas as pd
from tqdm import tqdm
from feat import Detector

from joblib import delayed, Parallel
from torchvision.utils import save_image
from torchvision.io import read_image, read_video
from torch.utils.data import Dataset
from feat.transforms import Rescale
import glob
from skimage.feature import hog
import pickle
from torch.utils.data import DataLoader
from feat.data import (
    Fex,
    ImageDataset,
    VideoDataset,
    _inverse_face_transform,
    _inverse_landmark_transform,
)
from feat.utils.image_operations import (
    extract_face_from_landmarks,
    extract_face_from_bbox,
    convert_image_to_tensor,
    BBox,
)

from tqdm import tqdm
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
import pickle

import matplotlib.pyplot as plt
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import StratifiedKFold

from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from imblearn import FunctionSampler
from xgboost import XGBClassifier
import shutil
import pickle
%matplotlib inline

In [2]:
df = pd.read_csv("pyfeats.csv")
df.loc[:,"input"] = df["input"].str.replace("mirrored_faces/","")

# correct mirror face left and right
orig_left_idx = df.input.str.contains('mirror_face_left')
orig_right_idx = df.input.str.contains('mirror_face_right')
mirror_face_df = df.loc[orig_left_idx, :]
df.loc[orig_left_idx,"input"] = mirror_face_df["input"].str.replace("left","right")
mirror_face_df = df.loc[orig_right_idx, :]
df.loc[orig_right_idx,"input"]= mirror_face_df["input"].str.replace("right","left")

# correct mirror head left and right
orig_left_idx = df.input.str.contains('mirror_head_left')
orig_right_idx = df.input.str.contains('mirror_head_right')
mirror_head_df = df.loc[orig_left_idx, :]
df.loc[orig_left_idx,"input"] = mirror_head_df["input"].str.replace("left","right")
mirror_head_df = df.loc[orig_right_idx, :]
df.loc[orig_right_idx,"input"]= mirror_head_df["input"].str.replace("right","left")
df = df.drop(columns=["frame","frame.1"])

In [3]:
mirror_head_df = df.loc[df.input.str.contains("mirror_head"),:]
mirror_head_df

,AU01,AU02,AU04,AU05,AU06,AU07,AU09,AU10,AU11,AU12,...,AU15,AU17,AU20,AU23,AU24,AU25,AU26,AU28,AU43,input
5,0.799944,0.600076,0.190919,0.736201,0.045757,0.0,0.136841,0.001946,1.0,0.039553,...,0.270311,0.621246,0.0,0.702713,0.395200,0.992278,0.804194,0.205958,0.048561,N_0000000001_00016-mirror_head_right.png
6,0.779600,0.522485,0.185228,0.742628,0.060773,0.0,0.101246,0.002326,1.0,0.032372,...,0.376858,0.339704,0.0,0.504433,0.335072,0.948967,0.486612,0.075014,0.058206,N_0000000001_00016-mirror_head_left.png
14,0.255167,0.232617,0.616801,0.393366,0.098053,1.0,0.089239,0.132092,1.0,0.057920,...,0.079623,0.557791,0.0,0.462376,0.444505,0.252814,0.450755,0.101466,0.024186,N_0000000001_00018-mirror_head_right.png
15,0.160417,0.053354,0.375736,0.416565,0.130824,0.0,0.119918,0.013823,1.0,0.041318,...,0.121289,0.396282,0.0,0.330781,0.454251,0.065559,0.018319,0.126827,0.023305,N_0000000001_00018-mirror_head_left.png
21,0.315658,0.038208,0.516897,0.358598,0.045530,0.0,0.136957,0.056062,0.0,0.013171,...,0.132860,0.374978,0.0,0.081197,0.284508,0.845008,0.320900,0.079929,0.049470,N_0000000001_00024-mirror_head_right.png
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36652,0.403982,0.152395,0.540337,0.384039,0.094641,0.0,0.177863,0.019368,1.0,0.047937,...,0.452046,0.415227,0.0,0.199519,0.115274,0.792827,0.154051,0.165678,0.041432,N_0000000042_00851-mirror_head_left.png
36658,0.473695,0.675528,0.032346,0.385015,0.138438,0.0,0.206454,0.004254,1.0,0.261735,...,0.103509,0.551068,0.0,0.449587,0.577684,0.025673,0.181472,0.580749,0.758482,N_0000000042_00852-mirror_head_right.png
36659,0.460230,0.615545,0.031520,0.453885,0.281705,0.0,0.237089,0.005991,0.0,0.611856,...,0.193468,0.443730,0.0,0.209124,0.239620,0.141872,0.188202,0.104658,0.275592,N_0000000042_00852-mirror_head_left.png
36665,0.869951,0.589591,0.362676,0.229095,0.147463,0.0,0.266634,0.091156,0.0,0.085085,...,0.703646,0.736167,0.0,0.702197,0.589688,0.003187,0.316437,0.628658,0.881412,N_0000000042_00854-mirror_head_right.png


In [4]:
mirror_face_df = df.loc[df.input.str.contains("mirror_face"),:]
mirror_face_df

,AU01,AU02,AU04,AU05,AU06,AU07,AU09,AU10,AU11,AU12,...,AU15,AU17,AU20,AU23,AU24,AU25,AU26,AU28,AU43,input
3,0.841569,0.726640,0.220696,0.696307,0.040540,0.0,0.085734,0.013668,1.0,0.025188,...,0.125362,0.393983,0.0,0.714709,0.237874,0.972804,0.760994,0.145093,0.041886,N_0000000001_00016-mirror_face_right.png
4,0.855943,0.649970,0.316961,0.745237,0.046467,0.0,0.100669,0.000825,0.0,0.023530,...,0.293480,0.578360,0.0,0.735081,0.568569,0.773202,0.689766,0.225282,0.046872,N_0000000001_00016-mirror_face_left.png
12,0.228380,0.124436,0.418362,0.389722,0.116291,0.0,0.122290,0.004784,0.0,0.045536,...,0.038212,0.410681,0.0,0.305951,0.326650,0.192757,0.151597,0.031994,0.031145,N_0000000001_00018-mirror_face_right.png
13,0.192308,0.167759,0.514176,0.395405,0.128107,0.0,0.088726,0.007669,1.0,0.052897,...,0.044137,0.372215,0.0,0.184544,0.301673,0.124226,0.078288,0.043802,0.025072,N_0000000001_00018-mirror_face_left.png
19,0.389544,0.118998,0.393467,0.241266,0.078988,0.0,0.221786,0.002430,0.0,0.020771,...,0.182489,0.347426,0.0,0.126244,0.113437,0.965032,0.190919,0.039208,0.536978,N_0000000001_00024-mirror_face_right.png
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36650,0.213645,0.094996,0.477487,0.356284,0.093895,0.0,0.123574,0.199212,1.0,0.024570,...,0.279829,0.461249,0.0,0.370944,0.153081,0.535168,0.072521,0.105007,0.072506,N_0000000042_00851-mirror_face_left.png
36656,0.351838,0.539636,0.014923,0.375043,0.247412,0.0,0.271209,0.034183,0.0,0.538099,...,0.122382,0.498420,0.0,0.282684,0.339361,0.553765,0.080913,0.109813,0.824501,N_0000000042_00852-mirror_face_right.png
36657,0.527599,0.598951,0.012107,0.369518,0.326397,0.0,0.195151,0.004242,0.0,0.451734,...,0.142726,0.476980,0.0,0.341333,0.214818,0.606268,0.273242,0.261104,0.232954,N_0000000042_00852-mirror_face_left.png
36663,0.744418,0.505726,0.409402,0.304247,0.324161,0.0,0.350389,0.042005,0.0,0.173190,...,0.349609,0.657380,0.0,0.838575,0.785200,0.086939,0.114774,0.151528,0.896369,N_0000000042_00854-mirror_face_right.png


In [5]:
unique_images = list(set(mirror_face_df["input"].str.replace("-mirror_face_right.png","").str.replace("-mirror_face_left.png","")))

C:\Users\Daniel\AppData\Local\Temp\ipykernel_17268\722240388.py:1: FutureWarning: The default value of regex will change from True to False in a future version.
  unique_images = list(set(mirror_face_df["input"].str.replace("-mirror_face_right.png","").str.replace("-mirror_face_left.png","")))


In [6]:
row_list = []
for im in unique_images:
    y_right = mirror_face_df[mirror_face_df["input"] == im+"-mirror_face_right.png"]
    y_right = y_right.rename(columns={x : x+"_right" for x in y_right.columns[:-1]})
    y_left = mirror_face_df[mirror_face_df["input"] == im+"-mirror_face_left.png"]
    y_left = y_left.rename(columns={x : x+"_left" for x in y_left.columns[:-1]})
    new_row = pd.concat([y_left.reset_index(),y_right.reset_index()], axis=1)
    new_row = new_row.drop(columns=["index", "input"])
    new_row["input"] = im
    row_list.append(new_row)

mirror_face_uni_df = pd.concat(row_list).reset_index().drop(columns=["index"])
mirror_face_uni_df

,AU01_left,AU02_left,AU04_left,AU05_left,AU06_left,AU07_left,AU09_left,AU10_left,AU11_left,AU12_left,...,AU15_right,AU17_right,AU20_right,AU23_right,AU24_right,AU25_right,AU26_right,AU28_right,AU43_right,input
0,0.578766,0.489559,0.261787,0.325537,0.585008,0.0,0.309508,0.512203,0.0,0.641765,...,0.339590,0.567035,0.0,0.511549,0.599525,0.700030,0.740559,0.141146,0.044334,N_0000000010_00434
1,0.326886,0.335751,0.390437,0.573323,0.130673,0.0,0.131025,0.035441,0.0,0.058132,...,0.102155,0.339337,1.0,0.291162,0.105315,0.998956,0.641708,0.019644,0.101597,N_0000000040_00602
2,0.473166,0.314336,0.126373,0.341674,0.545208,0.0,0.152294,0.113101,1.0,0.674304,...,0.541806,0.646596,0.0,0.371811,0.556727,0.082015,0.051038,0.420686,0.017902,N_0000000008_00364
3,0.345885,0.266269,0.129211,0.295090,0.913700,1.0,0.602774,0.988253,1.0,0.980239,...,0.073492,0.342141,1.0,0.054339,0.041868,0.997547,0.630570,0.100049,0.406931,N_0000000020_00714
4,0.573158,0.367996,0.345454,0.386497,0.212483,1.0,0.114682,0.153465,1.0,0.237111,...,0.294684,0.648442,0.0,0.374028,0.455633,0.004026,0.066143,0.142481,0.038039,N_0000000014_00144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5234,0.501116,0.202676,0.260051,0.409891,0.080218,0.0,0.103650,0.006144,0.0,0.033512,...,0.297232,0.534062,0.0,0.432854,0.605943,0.057370,0.100159,0.098918,0.100633,N_0000000006_00280
5235,0.333735,0.501258,0.217140,0.266113,0.414926,1.0,0.200047,0.189254,1.0,0.244506,...,0.321483,0.441171,0.0,0.592580,0.900141,0.020026,0.139743,0.365378,0.251912,N_0000000042_00237
5236,0.124236,0.261663,0.235967,0.330237,0.673577,1.0,0.481478,0.467108,0.0,0.876180,...,0.145955,0.263026,1.0,0.311355,0.396903,0.946759,0.254545,0.560218,0.033406,N_0000000027_00578
5237,0.846654,0.809987,0.065475,0.643402,0.604724,1.0,0.306887,0.997772,0.0,0.902812,...,0.366985,0.658713,0.0,0.720866,0.583332,0.031790,0.081927,0.110983,0.068096,N_0000000019_00560


In [7]:
mirror_face_uni_df.to_csv("mirror_face_uni_df.csv",index=None)

In [8]:
row_list = []
for im in unique_images:
    y_right = mirror_head_df[mirror_head_df["input"] == im+"-mirror_head_right.png"]
    y_right = y_right.rename(columns={x : x+"_right" for x in y_right.columns[:-1]})
    y_left = mirror_head_df[mirror_head_df["input"] == im+"-mirror_head_left.png"]
    y_left = y_left.rename(columns={x : x+"_left" for x in y_left.columns[:-1]})
    new_row = pd.concat([y_left.reset_index(),y_right.reset_index()], axis=1)
    new_row = new_row.drop(columns=["index", "input"])
    new_row["input"] = im
    row_list.append(new_row)

mirror_head_uni_df = pd.concat(row_list).reset_index().drop(columns=["index"])
mirror_head_uni_df

,AU01_left,AU02_left,AU04_left,AU05_left,AU06_left,AU07_left,AU09_left,AU10_left,AU11_left,AU12_left,...,AU15_right,AU17_right,AU20_right,AU23_right,AU24_right,AU25_right,AU26_right,AU28_right,AU43_right,input
0,0.565105,0.439607,0.143013,0.355727,0.660066,0.0,0.262484,0.283039,0.0,0.633705,...,0.299903,0.578427,0.0,0.604462,0.584747,0.910847,0.696021,0.150102,0.176989,N_0000000010_00434
1,0.321223,0.209483,0.400858,0.472276,0.187304,0.0,0.127408,0.044984,0.0,0.046384,...,0.183793,0.307559,1.0,0.234869,0.078276,0.984382,0.460940,0.007654,0.107521,N_0000000040_00602
2,0.402274,0.223611,0.077714,0.313061,0.632522,0.0,0.164265,0.238397,1.0,0.695138,...,0.600467,0.628896,0.0,0.466292,0.531503,0.029595,0.106436,0.283046,0.021038,N_0000000008_00364
3,0.438095,0.278681,0.122655,0.289265,0.898733,1.0,0.602909,0.950054,1.0,0.980660,...,0.346458,0.170380,1.0,0.195699,0.049228,0.990582,0.569130,0.034573,0.579909,N_0000000020_00714
4,0.519863,0.360507,0.207944,0.632381,0.175945,0.0,0.129922,0.026508,0.0,0.163705,...,0.230995,0.649949,0.0,0.366049,0.472652,0.003297,0.124334,0.204719,0.022753,N_0000000014_00144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5234,0.304360,0.081927,0.126988,0.330021,0.115197,0.0,0.152108,0.010199,0.0,0.057878,...,0.543769,0.492923,0.0,0.409646,0.621398,0.040554,0.166427,0.262843,0.030764,N_0000000006_00280
5235,0.307344,0.596315,0.200718,0.274648,0.357907,1.0,0.271145,0.439382,0.0,0.270037,...,0.196505,0.479545,1.0,0.691322,0.816925,0.785303,0.169825,0.533303,0.130254,N_0000000042_00237
5236,0.134583,0.336129,0.323927,0.311267,0.751268,1.0,0.440197,0.721662,0.0,0.951672,...,0.312371,0.286367,1.0,0.499257,0.139335,0.876241,0.485400,0.645489,0.051711,N_0000000027_00578
5237,0.816565,0.730443,0.065758,0.675381,0.480390,1.0,0.266496,0.997189,0.0,0.751090,...,0.235690,0.648612,0.0,0.592636,0.558429,0.015080,0.190550,0.124943,0.073798,N_0000000019_00560


In [9]:
mirror_head_uni_df.to_csv("mirror_head_uni_df.csv",index=None)

In [10]:
palsy_face_mirrored_df = pd.read_csv("palsy_face_mirrored_pyfeats.csv")
palsy_head_mirrored_df = pd.read_csv("palsy_head_mirrored_pyfeats.csv")

In [11]:
palsy_head_mirrored_df = palsy_head_mirrored_df.loc[:,["input", 'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10', 'AU11',
       'AU12', 'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24', 'AU25', 'AU26',
       'AU28', 'AU43']]
palsy_head_mirrored_df.loc[:,"input"] = palsy_head_mirrored_df["input"].str.replace("\\","/").str.replace("palsy_mirrored_faces/palsy_mirrored_head_corrected/","")
palsy_head_mirrored_df

C:\Users\Daniel\AppData\Local\Temp\ipykernel_17268\2556414317.py:4: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  palsy_head_mirrored_df.loc[:,"input"] = palsy_head_mirrored_df["input"].str.replace("\\","/").str.replace("palsy_mirrored_faces/palsy_mirrored_head_corrected/","")


,input,AU01,AU02,AU04,AU05,AU06,AU07,AU09,AU10,AU11,...,AU14,AU15,AU17,AU20,AU23,AU24,AU25,AU26,AU28,AU43
0,CompleteFlaccid1_1-mirror_head_right.png,0.557703,0.508785,0.097457,0.399308,0.137640,0.0,0.145723,0.007882,1.0,...,0.325006,0.585870,0.566303,0.0,0.585386,0.485501,0.004124,0.068827,0.076461,0.017684
1,CompleteFlaccid1_1-mirror_head_left.png,0.157627,0.167317,0.084624,0.428158,0.054941,0.0,0.109487,0.001139,0.0,...,0.075376,0.613580,0.511493,0.0,0.218641,0.609395,0.004214,0.052022,0.015908,0.018855
2,CompleteFlaccid1_2-mirror_head_right.png,0.939782,0.827301,0.057707,0.632256,0.097451,0.0,0.070419,0.001266,1.0,...,0.133530,0.538494,0.625045,0.0,0.411267,0.612786,0.001448,0.036807,0.266946,0.051237
3,CompleteFlaccid1_2-mirror_head_left.png,0.224377,0.160031,0.064268,0.451392,0.056800,0.0,0.114109,0.002127,0.0,...,0.148658,0.445318,0.510922,0.0,0.202780,0.662856,0.001499,0.046045,0.031810,0.021636
4,CompleteFlaccid1_3-mirror_head_right.png,0.650897,0.792430,0.579608,0.323281,0.260755,0.0,0.503899,0.087690,1.0,...,0.420698,0.608628,0.630524,0.0,0.475102,0.607039,0.004059,0.189185,0.104129,0.941849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,Normal9_6-mirror_head_left.png,0.658518,0.764349,0.132908,0.382876,0.821394,1.0,0.443547,0.983758,1.0,...,0.352698,0.264955,0.490834,1.0,0.105249,0.373915,0.990477,0.348620,0.111524,0.154109
956,Normal9_7-mirror_head_right.png,0.702850,0.562445,0.306770,0.534465,0.065852,0.0,0.110081,0.032880,0.0,...,0.091326,0.424287,0.416561,0.0,0.703559,0.280475,0.595675,0.436238,0.203850,0.022758
957,Normal9_7-mirror_head_left.png,0.791679,0.820204,0.235618,0.398718,0.156535,0.0,0.099552,0.013486,0.0,...,0.214218,0.446009,0.450295,0.0,0.711593,0.335762,0.409354,0.494175,0.682383,0.162774
958,Normal9_8-mirror_head_right.png,0.721719,0.716017,0.302255,0.458946,0.243958,0.0,0.160362,0.871506,0.0,...,0.111327,0.631559,0.204226,1.0,0.138072,0.012172,0.995399,0.891213,0.011652,0.029357


In [12]:
palsy_face_mirrored_df= palsy_face_mirrored_df.loc[:,["input", 'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10', 'AU11',
       'AU12', 'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24', 'AU25', 'AU26',
       'AU28', 'AU43']]
palsy_face_mirrored_df.loc[:,"input"] = palsy_face_mirrored_df["input"].str.replace("\\","/").str.replace("palsy_mirrored_faces/palsy_mirrored_face_corrected/","")
palsy_face_mirrored_df

C:\Users\Daniel\AppData\Local\Temp\ipykernel_17268\3068922145.py:4: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  palsy_face_mirrored_df.loc[:,"input"] = palsy_face_mirrored_df["input"].str.replace("\\","/").str.replace("palsy_mirrored_faces/palsy_mirrored_face_corrected/","")


,input,AU01,AU02,AU04,AU05,AU06,AU07,AU09,AU10,AU11,...,AU14,AU15,AU17,AU20,AU23,AU24,AU25,AU26,AU28,AU43
0,CompleteFlaccid1_1-mirror_face_right.png,0.590186,0.552490,0.052660,0.476747,0.079830,0.0,0.122039,0.015419,0.0,...,0.249794,0.339900,0.481138,0.0,0.457500,0.437431,0.015415,0.034388,0.011271,0.016915
1,CompleteFlaccid1_1-mirror_face_left.png,0.147366,0.247196,0.080439,0.386830,0.087016,0.0,0.080980,0.004369,0.0,...,0.131369,0.336791,0.503851,0.0,0.124224,0.527306,0.000565,0.061244,0.091767,0.024640
2,CompleteFlaccid1_2-mirror_face_right.png,0.892784,0.741301,0.035975,0.580008,0.068886,0.0,0.087801,0.006238,0.0,...,0.149187,0.298136,0.494949,0.0,0.186898,0.297255,0.027732,0.181040,0.016747,0.030334
3,CompleteFlaccid1_2-mirror_face_left.png,0.243040,0.273229,0.046470,0.480779,0.068603,0.0,0.076020,0.032236,1.0,...,0.102060,0.428861,0.560905,0.0,0.196133,0.505337,0.010482,0.084335,0.047774,0.013689
4,CompleteFlaccid1_3-mirror_face_right.png,0.670379,0.673394,0.702973,0.245923,0.193484,0.0,0.587743,0.036782,1.0,...,0.614310,0.400726,0.577779,0.0,0.382891,0.632761,0.011914,0.192978,0.074682,0.908171
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,Normal9_6-mirror_face_left.png,0.663565,0.783889,0.052823,0.351355,0.862143,1.0,0.368740,0.973469,1.0,...,0.384792,0.238582,0.421159,1.0,0.119295,0.214446,0.990835,0.372055,0.273532,0.118581
956,Normal9_7-mirror_face_right.png,0.813113,0.613408,0.323597,0.728855,0.054124,0.0,0.124730,0.009535,0.0,...,0.180672,0.480400,0.380825,0.0,0.731424,0.447315,0.435369,0.360297,0.809889,0.057297
957,Normal9_7-mirror_face_left.png,0.790696,0.512000,0.424883,0.418075,0.110194,0.0,0.107810,0.003011,0.0,...,0.269116,0.132776,0.545820,0.0,0.730715,0.729088,0.711535,0.639735,0.405600,0.071326
958,Normal9_8-mirror_face_right.png,0.721655,0.448606,0.426361,0.462961,0.162285,0.0,0.251767,0.197752,0.0,...,0.232683,0.534676,0.318865,1.0,0.198538,0.074433,0.995553,0.686376,0.041950,0.080778


In [13]:
palsy_images_list = list(set(palsy_face_mirrored_df["input"].str.replace("\\","/").str.replace("palsy_mirrored_faces/palsy_mirrored_face_corrected/","").str.replace("-mirror_face_right.png","").str.replace("-mirror_face_left.png","")))
palsy_images_list

C:\Users\Daniel\AppData\Local\Temp\ipykernel_17268\2685778937.py:1: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  palsy_images_list = list(set(palsy_face_mirrored_df["input"].str.replace("\\","/").str.replace("palsy_mirrored_faces/palsy_mirrored_face_corrected/","").str.replace("-mirror_face_right.png","").str.replace("-mirror_face_left.png","")))
C:\Users\Daniel\AppData\Local\Temp\ipykernel_17268\2685778937.py:1: FutureWarning: The default value of regex will change from True to False in a future version.
  palsy_images_list = list(set(palsy_face_mirrored_df["input"].str.replace("\\","/").str.replace("palsy_mirrored_faces/palsy_mirrored_face_corrected/","").str.replace("-mirror_face_right.png","").str.replace("-mirror_face_left.png","")))


['Synkinetic_NearNormal5_1',
 'Normal5_7',
 'CompleteFlaccid3_8',
 'CompleteFlaccid3_7',
 'Synkinetic_Severe1_1',
 'CompleteFlaccid1_8',
 'MildFlaccid2_2',
 'CompleteFlaccid5_1',
 'ModerateFlaccid1_8',
 'SevereFlaccid5_6',
 'Synkinetic_Complete5_5',
 'SevereFlaccid2_3',
 'Synkinetic_NearNormal4_1',
 'Normal8_1',
 'CompleteFlaccid2_3',
 'Normal3_8',
 'Normal2_5',
 'SevereFlaccid1_2',
 'CompleteFlaccid5_4',
 'Synkinetic_Complete4_8',
 'CompleteFlaccid2_8',
 'ModerateFlaccid4_5',
 'NearNormalFlaccid1_5',
 'Normal6_6',
 'Normal5_3',
 'Synkinetic_NearNormal3_5',
 'MildFlaccid3_4',
 'ModerateFlaccid1_3',
 'ModerateFlaccid3_7',
 'Synkinetic_NearNormal1_5',
 'Synkinetic_NearNormal4_2',
 'SevereFlaccid4_8',
 'NearNormalFlaccid3_7',
 'Synkinetic_Severe2_7',
 'Normal2_2',
 'Normal1_2',
 'Synkinetic_Complete1_8',
 'Synkinetic_NearNormal2_2',
 'Normal1_1',
 'Synkinetic_Mild1_3',
 'CompleteFlaccid4_4',
 'NearNormalFlaccid1_1',
 'NearNormalFlaccid5_1',
 'CompleteFlaccid3_5',
 'SevereFlaccid1_5',
 'Sy

In [14]:
row_list = []
for im in palsy_images_list:
    y_right = palsy_head_mirrored_df[palsy_head_mirrored_df["input"] == im+"-mirror_head_right.png"]
    y_right = y_right.rename(columns={x : x+"_right" for x in y_right.columns[1:]})
    y_left = palsy_head_mirrored_df[palsy_head_mirrored_df["input"] == im+"-mirror_head_left.png"]
    y_left = y_left.rename(columns={x : x+"_left" for x in y_left.columns[1:]})
    new_row = pd.concat([y_left.reset_index(),y_right.reset_index()], axis=1)
    new_row = new_row.drop(columns=["index", "input"])
    new_row["input"] = im
    row_list.append(new_row)

palsy_mirror_head_uni_df = pd.concat(row_list).reset_index().drop(columns=["index"])
palsy_mirror_head_uni_df

,AU01_left,AU02_left,AU04_left,AU05_left,AU06_left,AU07_left,AU09_left,AU10_left,AU11_left,AU12_left,...,AU15_right,AU17_right,AU20_right,AU23_right,AU24_right,AU25_right,AU26_right,AU28_right,AU43_right,input
0,0.477654,0.205698,0.146719,0.428166,0.121595,0.0,0.107640,0.016577,0.0,0.086242,...,0.641164,0.709794,0.0,0.690093,0.656437,0.216800,0.317152,0.791350,0.032822,Synkinetic_NearNormal5_1
1,0.222212,0.206948,0.182604,0.340731,0.067182,0.0,0.081807,0.001067,1.0,0.040367,...,0.208765,0.586653,0.0,0.442442,0.358188,0.603379,0.254509,0.572081,0.029335,Normal5_7
2,0.885129,0.761024,0.076583,0.460874,0.151290,0.0,0.135944,0.289897,0.0,0.206673,...,0.548932,0.537836,0.0,0.327761,0.206149,0.024156,0.089753,0.162128,0.038295,CompleteFlaccid3_8
3,0.821331,0.811601,0.169092,0.525905,0.461540,0.0,0.196277,0.041844,1.0,0.438600,...,0.284705,0.401482,0.0,0.419017,0.125128,0.064549,0.177928,0.032793,0.032773,CompleteFlaccid3_7
4,0.735026,0.814497,0.034735,0.476198,0.126154,0.0,0.069928,0.038460,1.0,0.079637,...,0.600618,0.648107,0.0,0.259757,0.451079,0.001085,0.126057,0.122677,0.073078,Synkinetic_Severe1_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,0.181416,0.498901,0.275292,0.424448,0.493883,0.0,0.194686,0.841209,1.0,0.398594,...,0.460172,0.641799,0.0,0.322276,0.339549,0.203772,0.101491,0.228636,0.023958,ModerateFlaccid4_6
476,0.873888,0.702395,0.191421,0.371483,0.485204,0.0,0.156125,0.236967,1.0,0.721254,...,0.434589,0.628861,0.0,0.489392,0.625812,0.032406,0.187685,0.081905,0.077148,Synkinetic_Complete4_2
477,0.299258,0.205057,0.589199,0.378130,0.166598,1.0,0.103853,0.009016,1.0,0.096118,...,0.116950,0.382672,0.0,0.296315,0.090664,0.099893,0.410019,0.602861,0.034795,Synkinetic_Moderate1_7
478,0.170852,0.209474,0.481550,0.285959,0.202573,0.0,0.232807,0.018222,1.0,0.033755,...,0.406238,0.503106,0.0,0.641316,0.484063,0.043371,0.108128,0.661077,0.730150,CompleteFlaccid1_4


In [15]:
row_list = []
for im in palsy_images_list:
    y_right = palsy_face_mirrored_df[palsy_face_mirrored_df["input"] == im+"-mirror_face_right.png"]
    y_right = y_right.rename(columns={x : x+"_right" for x in y_right.columns[1:]})
    y_left = palsy_face_mirrored_df[palsy_face_mirrored_df["input"] == im+"-mirror_face_left.png"]
    y_left = y_left.rename(columns={x : x+"_left" for x in y_left.columns[1:]})
    new_row = pd.concat([y_left.reset_index(),y_right.reset_index()], axis=1)
    new_row = new_row.drop(columns=["index", "input"])
    new_row["input"] = im
    row_list.append(new_row)

palsy_mirror_face_uni_df = pd.concat(row_list).reset_index().drop(columns=["index"])
palsy_mirror_face_uni_df

,AU01_left,AU02_left,AU04_left,AU05_left,AU06_left,AU07_left,AU09_left,AU10_left,AU11_left,AU12_left,...,AU15_right,AU17_right,AU20_right,AU23_right,AU24_right,AU25_right,AU26_right,AU28_right,AU43_right,input
0,0.530722,0.323336,0.156199,0.449490,0.111656,0.0,0.148767,0.061040,0.0,0.081122,...,0.451424,0.731107,0.0,0.638124,0.657163,0.210276,0.396952,0.555472,0.042298,Synkinetic_NearNormal5_1
1,0.263890,0.469577,0.220019,0.465003,0.107204,0.0,0.082307,0.008714,1.0,0.115696,...,0.291028,0.615592,0.0,0.343717,0.481724,0.746457,0.283294,0.320854,0.031125,Normal5_7
2,0.886667,0.773532,0.100404,0.452101,0.182755,0.0,0.152839,0.036570,0.0,0.077112,...,0.253901,0.528571,0.0,0.380502,0.512787,0.693657,0.034608,0.127221,0.047864,CompleteFlaccid3_8
3,0.859517,0.774548,0.089548,0.400098,0.579879,0.0,0.195749,0.196530,1.0,0.473225,...,0.143038,0.361833,0.0,0.221793,0.061096,0.951618,0.106772,0.065252,0.044822,CompleteFlaccid3_7
4,0.622292,0.674139,0.027194,0.411307,0.109070,0.0,0.068452,0.008245,1.0,0.075305,...,0.644815,0.444899,0.0,0.255266,0.240488,0.020797,0.110314,0.159700,0.039505,Synkinetic_Severe1_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,0.129156,0.464553,0.137630,0.379582,0.437061,1.0,0.153504,0.321190,1.0,0.272865,...,0.303029,0.592324,0.0,0.216451,0.213571,0.054611,0.117996,0.076233,0.033916,ModerateFlaccid4_6
476,0.882342,0.773792,0.109760,0.332122,0.531816,1.0,0.170238,0.775200,0.0,0.682413,...,0.702730,0.552917,1.0,0.447475,0.520848,0.005583,0.288329,0.068159,0.061502,Synkinetic_Complete4_2
477,0.325494,0.230174,0.468468,0.338052,0.153186,1.0,0.123381,0.016629,1.0,0.062037,...,0.164227,0.391856,0.0,0.555717,0.203202,0.032494,0.084988,0.473721,0.043353,Synkinetic_Moderate1_7
478,0.226155,0.251461,0.243143,0.346587,0.142519,0.0,0.108849,0.019773,1.0,0.034314,...,0.421561,0.514088,0.0,0.271811,0.207656,0.004338,0.110991,0.291428,0.503319,CompleteFlaccid1_4


In [16]:
AU_list = palsy_mirror_face_uni_df.columns[:-1]
AU_list

Index(['AU01_left', 'AU02_left', 'AU04_left', 'AU05_left', 'AU06_left',
       'AU07_left', 'AU09_left', 'AU10_left', 'AU11_left', 'AU12_left',
       'AU14_left', 'AU15_left', 'AU17_left', 'AU20_left', 'AU23_left',
       'AU24_left', 'AU25_left', 'AU26_left', 'AU28_left', 'AU43_left',
       'AU01_right', 'AU02_right', 'AU04_right', 'AU05_right', 'AU06_right',
       'AU07_right', 'AU09_right', 'AU10_right', 'AU11_right', 'AU12_right',
       'AU14_right', 'AU15_right', 'AU17_right', 'AU20_right', 'AU23_right',
       'AU24_right', 'AU25_right', 'AU26_right', 'AU28_right', 'AU43_right'],
      dtype='object')

In [17]:
def compile_dataset(saved_hog_path, image_list, mode="all"):
    """compile the saved hog and landmark features 
    Args:
        saved_hog_path: where you saved the HOGs in the last section
        au_df: a pandas dataframe that contains filepaths and AU annotations

    Returns:
        np.stack(hog_feats): a numpy array of hog features 
        np.stack(land_feats): a numpy array of landmarks 
        au_df.iloc[all_valid, :]: a pandas df of AU annotations 
    """
        
    all_valid = [] # Which images are valid images detectable by Py-Feat?
    hog_feats = [] # Aggregated HOG Features
    land_feats = [] # Aggregated Landmark Features
#     all_o_filename = image_list # Filenames in the annotation file
    all_hog_fp = [saved_hog_path+os.path.basename(op).split('.')[0]+'.pkl' for op in image_list]
    valid_images = []
    for image in image_list:
        hog_fp = saved_hog_path+os.path.basename(image).split('.')[0]+'.pkl'
        with open(hog_fp, 'rb') as fp:
            hog_feat, hog_feat_right, hog_feat_left, new_lands = pickle.load(fp)
        if  (len(hog_feat_right) == 5408) and (len(hog_feat) == 5408) and (len(hog_feat_left) == 5408) and (new_lands.shape[0] == 68) and (new_lands.shape[1] == 2): # Restrict to valid HOGs
            valid_images.append(image)
            all_valid.append(True)
            if mode=="all":
                hog_feats.append(hog_feat)
            elif mode=="right":
                hog_feats.append(hog_feat_right)
            elif mode=="left":
                hog_feats.append(hog_feat_left)
            land_feats.append(new_lands)
        else:
            all_valid.append(False)

    return np.stack(hog_feats), np.stack(land_feats), valid_images, all_valid

In [18]:
def func_samp(x, y):
    return x, y

def _run_and_testData(clf, x_features, y_features, sampling_method='under', cv_n_splits=5):
    """
    This function runs n-fold cross-validation on the dataset
    """
    if sampling_method == 'under':
        ros = RandomUnderSampler(random_state=0)
    elif sampling_method == 'smote':
        ros1 = SMOTE(random_state=0, sampling_strategy=0.60)
        ros2 = RandomUnderSampler(random_state=0, sampling_strategy=0.5)
        ros = Pipeline(steps=[('o', ros1), ('u', ros2)])
    else:
        ros = FunctionSampler(func=func_samp)

    valid_train_idx = np.where(np.logical_or(y_features == 0, y_features==1))[0]
    x_training_valid, y_training_valid = x_features[valid_train_idx, :], y_features[valid_train_idx]
    
    
    
    
    skf = StratifiedKFold(n_splits=cv_n_splits, random_state=1, shuffle=True)
    
    prec_list = []
    rec_list = []
    fscore_list = []
    acc_list = []
    
    if x_training_valid.shape[0] > 0 :
        for i, (train_index, val_index) in enumerate(skf.split(X=x_training_valid, y=y_training_valid)):

            xx_train, xx_val = x_training_valid[train_index], x_training_valid[val_index]
            yy_train, yy_val = y_training_valid[train_index], y_training_valid[val_index]

            xx_bal, yy_bal = ros.fit_resample(xx_train, yy_train)
            clf.fit(xx_bal, yy_bal)
            fitted_pred = clf.predict(xx_bal)
            prec, rec, fscore, supp = precision_recall_fscore_support(y_true=yy_bal, y_pred=fitted_pred, average='binary')
    #         print('training score:', prec, rec, fscore)

            preds = clf.predict(xx_val)
            prec, rec, fscore, supp = precision_recall_fscore_support(y_true=yy_val, y_pred=preds, average='binary')
            acc = accuracy_score(y_true=yy_val, y_pred=preds)

            prec_list.append(prec)
            rec_list.append(rec)
            fscore_list.append(fscore)
            acc_list.append(acc)
        
#         print('validation score:', prec, rec, fscore, acc)
        
    return prec_list, rec_list, fscore_list, acc_list



def train_cv_savemodel(clf, x_features, y_features, name="", sampling_method='under', cv_n_splits=5):
    """
    This function runs n-fold cross-validation on the dataset
    """
    if sampling_method == 'under':
        ros = RandomUnderSampler(random_state=0)
    elif sampling_method == 'smote':
        ros1 = SMOTE(random_state=0, sampling_strategy=0.60)
        ros2 = RandomUnderSampler(random_state=0, sampling_strategy=0.5)
        ros = Pipeline(steps=[('o', ros1), ('u', ros2)])
    else:
        ros = FunctionSampler(func=func_samp)

    valid_train_idx = np.where(np.logical_or(y_features == 0, y_features==1))[0]
    x_training_valid, y_training_valid = x_features[valid_train_idx, :], y_features[valid_train_idx]
    
    
    skf = StratifiedKFold(n_splits=cv_n_splits, random_state=1, shuffle=True)
    
    prec_list = []
    rec_list = []
    fscore_list = []
    acc_list = []
    
    if x_training_valid.shape[0] > 0 :
        for i, (train_index, val_index) in enumerate(skf.split(X=x_training_valid, y=y_training_valid)):

            xx_train, xx_val = x_training_valid[train_index], x_training_valid[val_index]
            yy_train, yy_val = y_training_valid[train_index], y_training_valid[val_index]

            xx_bal, yy_bal = ros.fit_resample(xx_train, yy_train)
            clf.fit(xx_bal, yy_bal)
            fitted_pred = clf.predict(xx_bal)
            prec, rec, fscore, supp = precision_recall_fscore_support(y_true=yy_bal, y_pred=fitted_pred, average='binary')
    #         print('training score:', prec, rec, fscore)
            
            preds = clf.predict(xx_val)
            prec, rec, fscore, supp = precision_recall_fscore_support(y_true=yy_val, y_pred=preds, average='binary')
            acc = accuracy_score(y_true=yy_val, y_pred=preds)

            prec_list.append(prec)
            rec_list.append(rec)
            fscore_list.append(fscore)
            acc_list.append(acc)
            
#             os.makedirs("saved_weights", exist_ok=True)
#             with open("saved_weights/{}-fold-{}.pkl".format(name,i), "wb") as f:
#                 pickle.dump(clf, f)
#         print('validation score:', prec, rec, fscore, acc)
        
    return prec_list, rec_list, fscore_list, acc_list




def format_metric_mean_std(metric_list):
    if metric_list is None:
        return None
    
    if len(metric_list) == 0:
        return None
    
    mu = np.mean(metric_list) * 100
    std = np.std(metric_list) * 100
    return "{:0.2f} ± {:0.2f}".format(mu, std)
    

In [19]:
def prepare_data(saved_hog_path, image_list, feature_mode, scaler=None, pca=None):
    trained_hogs, trained_land, valid_images, valid_list = compile_dataset(saved_hog_path=saved_hog_path,
                                                            image_list=image_list, mode=feature_mode)
    trained_land = trained_land.reshape(trained_hogs.shape[0], -1)

    if scaler is None:
        scaler = StandardScaler()
        hog_data_full_std = scaler.fit_transform(trained_hogs)
    else:
        hog_data_full_std = scaler.transform(trained_hogs)
    if pca is None:
        pca = PCA(n_components=0.95)
        hog_data_full_transformed = pca.fit_transform(hog_data_full_std)
    else:
        hog_data_full_transformed = pca.transform(hog_data_full_std)
    
    x_features = np.concatenate([hog_data_full_transformed, trained_land], 1)
    return x_features, scaler, pca, valid_images, valid_list

In [20]:
feature_mode = "all"
data_mode = "half_faces"
SAVE_HOG_DIR = 'HOGFeatures/half_faces_all/'
x_train_features, scaler, pca, valid_images_train, valid_list = prepare_data(saved_hog_path=SAVE_HOG_DIR,image_list=mirror_face_uni_df["input"],feature_mode=feature_mode)


In [21]:
mirror_face_uni_df = mirror_face_uni_df[valid_list]

In [22]:
mirror_head_uni_df = mirror_head_uni_df[valid_list]

In [23]:
metrics_df = {x:pd.DataFrame(columns=["model"] + list(AU_list)) for x in ["prec", "rec", "fscore", "acc"]}

In [24]:
thresholds = [0.5,0.6,0.7,0.75,0.8]
thresholds

[0.5, 0.6, 0.7, 0.75, 0.8]

In [25]:
def run_train_model(thresholds, x_train_features, label_df, data_mode, feature_mode, ml_model="svm",metrics_df=None):
    np.random.seed(1)
    for thresh in thresholds:

        model_name = "{}_{}_{}_{:0.2f}".format(ml_model,data_mode, feature_mode, thresh)
        metrics_dict = {}
        metrics_dict.setdefault("prec", {})["model"] = model_name
        metrics_dict.setdefault("rec", {})["model"] = model_name
        metrics_dict.setdefault("fscore", {})["model"] = model_name
        metrics_dict.setdefault("acc", {})["model"] = model_name

        for AU in tqdm(AU_list):
            prec_list = []
            rec_list = []
            fscore_list = []
            acc_list = []
            y_features = label_df[AU].to_numpy() >= thresh

            if (y_features == True).sum() > 10 and (y_features == False).sum() > 10:
                if ml_model == "svm":

                    model_AU = LinearSVC(penalty='l2', C=5e-5, loss='squared_hinge', tol=2e-4, max_iter=2000, dual="auto")
                else:
                    model_AU = XGBClassifier(device="gpu")
                    
                prec_list, rec_list, fscore_list, acc_list = train_cv_savemodel(clf=model_AU,
                                  x_features=x_train_features, y_features=y_features, name=model_name, sampling_method='under', cv_n_splits=5)

            prec = format_metric_mean_std(prec_list)
            rec = format_metric_mean_std(rec_list)
            fscore = format_metric_mean_std(fscore_list)
            acc = format_metric_mean_std(acc_list)

            metrics_dict.setdefault("prec", {})[AU] = prec
            metrics_dict.setdefault("rec", {})[AU] = rec
            metrics_dict.setdefault("fscore", {})[AU] = fscore
            metrics_dict.setdefault("acc", {})[AU] = acc
        for metric in metrics_dict:
            metrics_df[metric] = metrics_df[metric].append(metrics_dict[metric], ignore_index=True)

# Train all no cross validation (Palsy)


In [91]:
metrics_df_palsy = {x:pd.DataFrame(columns=["model"] + list(AU_list)) for x in ["prec", "rec", "fscore", "acc"]}

In [50]:
feature_mode = "all"
data_mode = "half_faces"
SAVE_HOG_DIR = 'HOGFeatures/half_faces_all/'
x_train_features, scaler, pca, valid_images_train, valid_list = prepare_data(saved_hog_path=SAVE_HOG_DIR,image_list=mirror_face_uni_df["input"],feature_mode=feature_mode)


In [51]:
mirror_face_uni_df = mirror_face_uni_df[valid_list]

In [52]:
mirror_head_uni_df = mirror_head_uni_df[valid_list]

In [67]:
x_palsy_face_features, _, _, valid_images_val, palsy_valid_list = prepare_data(saved_hog_path="HOGFeatures/palsy_original/", image_list=palsy_mirror_face_uni_df["input"], feature_mode=feature_mode, scaler=scaler,pca=pca)

In [68]:
palsy_mirror_face_uni_df = palsy_mirror_face_uni_df[palsy_valid_list]

In [69]:
palsy_mirror_head_uni_df = palsy_mirror_head_uni_df[palsy_valid_list]

In [70]:
palsy_mirror_face_uni_df

,AU01_left,AU02_left,AU04_left,AU05_left,AU06_left,AU07_left,AU09_left,AU10_left,AU11_left,AU12_left,...,AU15_right,AU17_right,AU20_right,AU23_right,AU24_right,AU25_right,AU26_right,AU28_right,AU43_right,input
0,0.912541,0.822716,0.263748,0.430815,0.110512,0.0,0.072251,0.004503,0.0,0.029639,...,0.529509,0.560990,0.0,0.093788,0.267686,0.011402,0.048400,0.261926,0.121967,CompleteFlaccid3_2
1,0.198415,0.141321,0.320213,0.282349,0.609776,1.0,0.636486,0.971824,1.0,0.568746,...,0.414760,0.487228,0.0,0.434565,0.535697,0.892324,0.325163,0.080023,0.750067,Normal3_4
2,0.539884,0.374989,0.602104,0.309893,0.345139,1.0,0.332587,0.142262,1.0,0.090117,...,0.785170,0.739747,0.0,0.760811,0.776507,0.031175,0.553928,0.565403,0.898696,Synkinetic_NearNormal5_3
3,0.306743,0.329371,0.100803,0.302152,0.216002,0.0,0.079907,0.150166,1.0,0.093069,...,0.177021,0.515146,0.0,0.760326,0.509547,0.650591,0.619740,0.931808,0.931830,Synkinetic_Moderate3_6
4,0.212882,0.132825,0.268583,0.268644,0.854335,1.0,0.543009,0.971512,1.0,0.905493,...,0.450973,0.585013,1.0,0.356421,0.552310,0.046154,0.091726,0.024690,0.102288,Synkinetic_NearNormal3_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,0.642857,0.437075,0.436015,0.272661,0.588938,1.0,0.545304,0.073770,0.0,0.623655,...,0.540158,0.624080,0.0,0.527005,0.424109,0.030661,0.210043,0.031395,0.894157,Synkinetic_Complete4_3
476,0.162253,0.086175,0.105383,0.217479,0.898758,1.0,0.472712,0.796528,1.0,0.946012,...,0.655828,0.417991,1.0,0.480383,0.014630,0.999348,0.734449,0.086592,0.045984,Synkinetic_Moderate1_6
477,0.205298,0.234176,0.132534,0.208233,0.779970,1.0,0.359324,0.438442,1.0,0.849766,...,0.699099,0.597892,0.0,0.646331,0.374378,0.973871,0.242780,0.229272,0.038954,Synkinetic_Moderate1_5
478,0.842993,0.780505,0.196695,0.222725,0.277635,1.0,0.285719,0.004648,0.0,0.044370,...,0.699620,0.612349,0.0,0.422755,0.640318,0.000701,0.364856,0.162328,0.847483,NearNormalFlaccid5_3


In [93]:
# mirror head gt
feature_mode = "all"
SAVE_HOG_DIR = 'HOGFeatures/half_faces_all/'
x_train_features, scaler, pca, valid_images_train, valid_list = prepare_data(saved_hog_path=SAVE_HOG_DIR,image_list=mirror_head_uni_df["input"],feature_mode=feature_mode)
for feature_mode in ["all", "left", "right"]:
    
    x_train_features, scaler, pca, valid_images_train, valid_list = prepare_data(saved_hog_path=SAVE_HOG_DIR,image_list=mirror_head_uni_df["input"],feature_mode=feature_mode)
    x_palsy_head_features, _, _, valid_images_val, palsy_valid_list = prepare_data(saved_hog_path="HOGFeatures/palsy_original/", image_list=palsy_mirror_head_uni_df["input"], feature_mode=feature_mode, scaler=scaler,pca=pca)
    
    for thresh in thresholds:
        model_name = "{}_{}_{}_{:0.2f}".format("svm","palsy_head", feature_mode, thresh)
        metrics_dict_svm = {}
        metrics_dict_svm.setdefault("prec", {})["model"] = model_name
        metrics_dict_svm.setdefault("rec", {})["model"] = model_name
        metrics_dict_svm.setdefault("fscore", {})["model"] = model_name
        metrics_dict_svm.setdefault("acc", {})["model"] = model_name


        model_name = "{}_{}_{}_{:0.2f}".format("xgb","palsy_head", feature_mode, thresh)
        metrics_dict_xgb = {}
        metrics_dict_xgb.setdefault("prec", {})["model"] = model_name
        metrics_dict_xgb.setdefault("rec", {})["model"] = model_name
        metrics_dict_xgb.setdefault("fscore", {})["model"] = model_name
        metrics_dict_xgb.setdefault("acc", {})["model"] = model_name
        for AU in tqdm(AU_list):
    #         prec_list = []
    #         rec_list = []
    #         fscore_list = []
    #         acc_list = []
            y_features = mirror_head_uni_df[AU].to_numpy() >= thresh
            
            if (y_features == True).sum() > 10 and (y_features == False).sum() > 10:
                model_AU_svm = LinearSVC(penalty='l2', C=5e-5, loss='squared_hinge', tol=2e-4, max_iter=2000, dual="auto")
                model_AU_xgb = XGBClassifier(device="gpu")


                ros = RandomUnderSampler(random_state=0)

                valid_train_idx = np.where(np.logical_or(y_features == 0, y_features==1))[0]
                x_training_valid, y_training_valid = x_train_features[valid_train_idx, :], y_features[valid_train_idx]

                xx_bal, yy_bal = ros.fit_resample(x_training_valid, y_training_valid)
                model_AU_svm.fit(xx_bal, yy_bal)
                model_AU_xgb.fit(xx_bal, yy_bal)

        #         fitted_pred = model_AU_svm.predict(xx_bal)
        #         prec, rec, fscore, supp = precision_recall_fscore_support(y_true=yy_bal, y_pred=fitted_pred, average='binary')
        #         acc = accuracy_score(y_true=yy_bal, y_pred=fitted_pred)

                y_palsy = palsy_mirror_head_uni_df[AU].to_numpy() >= thresh

                svm_fitted_pred = model_AU_svm.predict(x_palsy_head_features)
                xgb_fitted_pred = model_AU_xgb.predict(x_palsy_head_features)

                prec_svm, rec_svm, fscore_svm, supp = precision_recall_fscore_support(y_true=y_palsy, y_pred=svm_fitted_pred, average='binary')
                acc_svm = accuracy_score(y_true=y_palsy, y_pred=svm_fitted_pred)

                prec_xgb, rec_xgb, fscore_xgb, supp = precision_recall_fscore_support(y_true=y_palsy, y_pred=xgb_fitted_pred, average='binary')
                acc_xgb = accuracy_score(y_true=y_palsy, y_pred=xgb_fitted_pred)
            else:
                prec_svm = None
                rec_svm = None
                fscore_svm = None
                acc_svm = None
                prec_xgb = None
                rec_xgb = None
                fscore_xgb = None
                acc_xgb = None
            metrics_dict_svm.setdefault("prec", {})[AU] = prec_svm
            metrics_dict_svm.setdefault("rec", {})[AU] = rec_svm
            metrics_dict_svm.setdefault("fscore", {})[AU] = fscore_svm
            metrics_dict_svm.setdefault("acc", {})[AU] = acc_svm

            metrics_dict_xgb.setdefault("prec", {})[AU] = prec_xgb
            metrics_dict_xgb.setdefault("rec", {})[AU] = rec_xgb
            metrics_dict_xgb.setdefault("fscore", {})[AU] = fscore_xgb
            metrics_dict_xgb.setdefault("acc", {})[AU] = acc_xgb

        for metric in metrics_dict_svm:
            metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
        for metric in metrics_dict_xgb:
            metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_xgb[metric], ignore_index=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [01:10<00:00,  1.77s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [00:51<00:00,  1.30s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [00:47<00:00,  1.18s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [00:53<00:00,  1.34s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [00:38<00:00,  1.05it/s]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\2994756076.py:79: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

In [94]:
metrics_df_palsy

{'prec':                         model AU01_left AU02_left AU04_left AU05_left  \
 0     svm_palsy_face_all_0.30  0.848066   0.79822  0.623529  0.867742   
 1     xgb_palsy_face_all_0.30  0.831522  0.809375  0.538462  0.877622   
 2     svm_palsy_face_all_0.40  0.748428   0.69103  0.533784  0.685106   
 3     xgb_palsy_face_all_0.40  0.733728  0.710714  0.503106   0.71875   
 4     svm_palsy_face_all_0.50  0.650177   0.59176  0.485981      0.34   
 ..                        ...       ...       ...       ...       ...   
 67  xgb_palsy_head_right_0.60  0.462838  0.444444  0.308824   0.20979   
 68  svm_palsy_head_right_0.70  0.329502  0.370558  0.394366  0.059783   
 69  xgb_palsy_head_right_0.70  0.333333  0.422857  0.347826  0.074324   
 70  svm_palsy_head_right_0.80   0.20354  0.226131   0.34375      None   
 71  xgb_palsy_head_right_0.80  0.178439  0.189189  0.205882      None   
 
    AU06_left AU07_left AU09_left AU10_left AU11_left  ... AU14_right  \
 0   0.906077  0.825112  0.78

In [95]:
for metric in metrics_df_palsy:
    metrics_df_palsy[metric].to_csv("HOG_AU_palsy_eval_{}.csv".format(metric))

In [92]:
# mirror face gt
feature_mode = "all"
SAVE_HOG_DIR = 'HOGFeatures/half_faces_all/'
x_train_features, scaler, pca, valid_images_train, valid_list = prepare_data(saved_hog_path=SAVE_HOG_DIR,image_list=mirror_face_uni_df["input"],feature_mode=feature_mode)
for feature_mode in ["all", "left", "right"]:
    
    x_train_features, scaler, pca, valid_images_train, valid_list = prepare_data(saved_hog_path=SAVE_HOG_DIR,image_list=mirror_face_uni_df["input"],feature_mode=feature_mode)
    x_palsy_face_features, _, _, valid_images_val, palsy_valid_list = prepare_data(saved_hog_path="HOGFeatures/palsy_original/", image_list=palsy_mirror_face_uni_df["input"], feature_mode=feature_mode, scaler=scaler,pca=pca)

    for thresh in thresholds:
        model_name = "{}_{}_{}_{:0.2f}".format("svm","palsy_face", feature_mode, thresh)
        metrics_dict_svm = {}
        metrics_dict_svm.setdefault("prec", {})["model"] = model_name
        metrics_dict_svm.setdefault("rec", {})["model"] = model_name
        metrics_dict_svm.setdefault("fscore", {})["model"] = model_name
        metrics_dict_svm.setdefault("acc", {})["model"] = model_name


        model_name = "{}_{}_{}_{:0.2f}".format("xgb","palsy_face", feature_mode, thresh)
        metrics_dict_xgb = {}
        metrics_dict_xgb.setdefault("prec", {})["model"] = model_name
        metrics_dict_xgb.setdefault("rec", {})["model"] = model_name
        metrics_dict_xgb.setdefault("fscore", {})["model"] = model_name
        metrics_dict_xgb.setdefault("acc", {})["model"] = model_name
        for AU in tqdm(AU_list):
    #         prec_list = []
    #         rec_list = []
    #         fscore_list = []
    #         acc_list = []
            y_features = mirror_face_uni_df[AU].to_numpy() >= thresh
        
            
            if (y_features == True).sum() > 10 and (y_features == False).sum() > 10:

                model_AU_svm = LinearSVC(penalty='l2', C=5e-5, loss='squared_hinge', tol=2e-4, max_iter=2000, dual="auto")
                model_AU_xgb = XGBClassifier(device="gpu")


                ros = RandomUnderSampler(random_state=0)

                valid_train_idx = np.where(np.logical_or(y_features == 0, y_features==1))[0]
                x_training_valid, y_training_valid = x_train_features[valid_train_idx, :], y_features[valid_train_idx]

                xx_bal, yy_bal = ros.fit_resample(x_training_valid, y_training_valid)
                model_AU_svm.fit(xx_bal, yy_bal)
                model_AU_xgb.fit(xx_bal, yy_bal)

        #         fitted_pred = model_AU_svm.predict(xx_bal)
        #         prec, rec, fscore, supp = precision_recall_fscore_support(y_true=yy_bal, y_pred=fitted_pred, average='binary')
        #         acc = accuracy_score(y_true=yy_bal, y_pred=fitted_pred)

                y_palsy = palsy_mirror_face_uni_df[AU].to_numpy() >= thresh

                svm_fitted_pred = model_AU_svm.predict(x_palsy_face_features)
                xgb_fitted_pred = model_AU_xgb.predict(x_palsy_face_features)

                prec_svm, rec_svm, fscore_svm, supp = precision_recall_fscore_support(y_true=y_palsy, y_pred=svm_fitted_pred, average='binary')
                acc_svm = accuracy_score(y_true=y_palsy, y_pred=svm_fitted_pred)

                prec_xgb, rec_xgb, fscore_xgb, supp = precision_recall_fscore_support(y_true=y_palsy, y_pred=xgb_fitted_pred, average='binary')
                acc_xgb = accuracy_score(y_true=y_palsy, y_pred=xgb_fitted_pred)
            else:
                prec_svm = None
                rec_svm = None
                fscore_svm = None
                acc_svm = None
                prec_xgb = None
                rec_xgb = None
                fscore_xgb = None
                acc_xgb = None
            metrics_dict_svm.setdefault("prec", {})[AU] = prec_svm
            metrics_dict_svm.setdefault("rec", {})[AU] = rec_svm
            metrics_dict_svm.setdefault("fscore", {})[AU] = fscore_svm
            metrics_dict_svm.setdefault("acc", {})[AU] = acc_svm

            metrics_dict_xgb.setdefault("prec", {})[AU] = prec_xgb
            metrics_dict_xgb.setdefault("rec", {})[AU] = rec_xgb
            metrics_dict_xgb.setdefault("fscore", {})[AU] = fscore_xgb
            metrics_dict_xgb.setdefault("acc", {})[AU] = acc_xgb

        for metric in metrics_dict_svm:
            metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
        for metric in metrics_dict_xgb:
            metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_xgb[metric], ignore_index=True)

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [01:09<00:00,  1.73s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [00:48<00:00,  1.22s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [00:44<00:00,  1.12s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [00:52<00:00,  1.32s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

100%|██████████████████████████████████████████████████████████████████████████████████| 40/40 [00:36<00:00,  1.09it/s]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_31820\3130950013.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_palsy[metric] = metrics_df_palsy[metric].append(m

# Train no cross_val manual GT annotation

In [46]:
gt_df = pd.read_csv("gt_annotation.csv")
AU_list = gt_df.columns[1:]
gt_df

,image,AU02_left,AU04_left,AU10_left,AU15_left,AU25_left,AU26_left,AU28_left,AU43_left,AU02_right,AU04_right,AU10_right,AU15_right,AU25_right,AU26_right,AU28_right,AU43_right
0,N_0000000014_00442,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0,0
1,N_0000000010_00128,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,N_0000000005_00643,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,N_0000000002_00423,0,0,1,0,1,1,0,0,0,1,1,0,1,1,0,0
4,N_0000000006_00213,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,N_0000000022_00144,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0
60,N_0000000013_00017,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0
61,N_0000000009_00240,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
62,N_0000000009_00797,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [47]:
AU_left = [x for x in gt_df.columns if "left" in x]
AU_right = [x for x in gt_df.columns if "right" in x]

In [53]:
gt_df2 = gt_df.copy()
gt_df2["image"] = gt_df2["image"] +"-flipped"
gt_df2.loc[:,AU_left + AU_right] = gt_df2.loc[:, AU_right+AU_left].values
gt_flipped_df = pd.concat([gt_df, gt_df2],ignore_index=True)
gt_flipped_df

,image,AU02_left,AU04_left,AU10_left,AU15_left,AU25_left,AU26_left,AU28_left,AU43_left,AU02_right,AU04_right,AU10_right,AU15_right,AU25_right,AU26_right,AU28_right,AU43_right
0,N_0000000014_00442,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0,0
1,N_0000000010_00128,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,N_0000000005_00643,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,N_0000000002_00423,0,0,1,0,1,1,0,0,0,1,1,0,1,1,0,0
4,N_0000000006_00213,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,N_0000000022_00144-flipped,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0
124,N_0000000013_00017-flipped,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0
125,N_0000000009_00240-flipped,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
126,N_0000000009_00797-flipped,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


,image,AU02_left,AU04_left,AU10_left,AU15_left,AU25_left,AU26_left,AU28_left,AU43_left,AU02_right,AU04_right,AU10_right,AU15_right,AU25_right,AU26_right,AU28_right,AU43_right
0,N_0000000014_00442-flipped,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0
1,N_0000000010_00128-flipped,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,N_0000000005_00643-flipped,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,N_0000000002_00423-flipped,0,1,1,0,1,1,0,0,0,0,1,0,1,1,0,0
4,N_0000000006_00213-flipped,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,N_0000000022_00144-flipped,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0
60,N_0000000013_00017-flipped,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0
61,N_0000000009_00240-flipped,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
62,N_0000000009_00797-flipped,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [36]:
# remove images in training
mirror_head_uni_df = mirror_head_uni_df[~mirror_head_uni_df["input"].isin(gt_df["image"])]
thresholds

[0.5, 0.6, 0.7, 0.75, 0.8]

In [38]:
# mirror head gt
metrics_df_anno = {x:pd.DataFrame(columns=["model"] + list(AU_list)) for x in ["prec", "rec", "fscore", "acc"]}
feature_mode = "all"
SAVE_HOG_DIR = 'HOGFeatures/half_faces_all_flipped/'
# x_train_features, scaler, pca, valid_images_train, valid_list = prepare_data(saved_hog_path=SAVE_HOG_DIR,image_list=mirror_head_uni_df["input"],feature_mode=feature_mode)
for feature_mode in ["all", "left", "right"]:
    
    x_train_features, scaler, pca, valid_images_train, valid_list = prepare_data(saved_hog_path=SAVE_HOG_DIR,image_list=mirror_head_uni_df["input"],feature_mode=feature_mode)
    x_anno_head_features, _, _, valid_images_val, anno_valid_list = prepare_data(saved_hog_path="HOGFeatures/half_faces_all_flipped/", image_list=gt_df["image"], feature_mode=feature_mode, scaler=scaler,pca=pca)
    
    for thresh in thresholds:
        model_name = "{}_{}_{}_{:0.2f}".format("svm","anno_head", feature_mode, thresh)
        metrics_dict_svm = {}
        metrics_dict_svm.setdefault("prec", {})["model"] = model_name
        metrics_dict_svm.setdefault("rec", {})["model"] = model_name
        metrics_dict_svm.setdefault("fscore", {})["model"] = model_name
        metrics_dict_svm.setdefault("acc", {})["model"] = model_name


        model_name = "{}_{}_{}_{:0.2f}".format("xgb","anno_head", feature_mode, thresh)
        metrics_dict_xgb = {}
        metrics_dict_xgb.setdefault("prec", {})["model"] = model_name
        metrics_dict_xgb.setdefault("rec", {})["model"] = model_name
        metrics_dict_xgb.setdefault("fscore", {})["model"] = model_name
        metrics_dict_xgb.setdefault("acc", {})["model"] = model_name
        for AU in tqdm(AU_list):
    #         prec_list = []
    #         rec_list = []
    #         fscore_list = []
    #         acc_list = []
            y_features = mirror_head_uni_df[AU].to_numpy() >= thresh
            
            if (y_features == True).sum() > 10 and (y_features == False).sum() > 10:
                model_AU_svm = LinearSVC(penalty='l2', C=5e-5, loss='squared_hinge', tol=2e-4, max_iter=2000, dual="auto")
                model_AU_xgb = XGBClassifier(device="gpu")


                ros = RandomUnderSampler(random_state=0)

                valid_train_idx = np.where(np.logical_or(y_features == 0, y_features==1))[0]
                x_training_valid, y_training_valid = x_train_features[valid_train_idx, :], y_features[valid_train_idx]

                xx_bal, yy_bal = ros.fit_resample(x_training_valid, y_training_valid)
                model_AU_svm.fit(xx_bal, yy_bal)
                model_AU_xgb.fit(xx_bal, yy_bal)

        #         fitted_pred = model_AU_svm.predict(xx_bal)
        #         prec, rec, fscore, supp = precision_recall_fscore_support(y_true=yy_bal, y_pred=fitted_pred, average='binary')
        #         acc = accuracy_score(y_true=yy_bal, y_pred=fitted_pred)

                y_anno = gt_df[AU].to_numpy()

                svm_fitted_pred = model_AU_svm.predict(x_anno_head_features)
                xgb_fitted_pred = model_AU_xgb.predict(x_anno_head_features)

                prec_svm, rec_svm, fscore_svm, supp = precision_recall_fscore_support(y_true=y_anno, y_pred=svm_fitted_pred, average='binary')
                acc_svm = accuracy_score(y_true=y_anno, y_pred=svm_fitted_pred)

                prec_xgb, rec_xgb, fscore_xgb, supp = precision_recall_fscore_support(y_true=y_anno, y_pred=xgb_fitted_pred, average='binary')
                acc_xgb = accuracy_score(y_true=y_anno, y_pred=xgb_fitted_pred)
            else:
                prec_svm = None
                rec_svm = None
                fscore_svm = None
                acc_svm = None
                prec_xgb = None
                rec_xgb = None
                fscore_xgb = None
                acc_xgb = None
            metrics_dict_svm.setdefault("prec", {})[AU] = prec_svm
            metrics_dict_svm.setdefault("rec", {})[AU] = rec_svm
            metrics_dict_svm.setdefault("fscore", {})[AU] = fscore_svm
            metrics_dict_svm.setdefault("acc", {})[AU] = acc_svm

            metrics_dict_xgb.setdefault("prec", {})[AU] = prec_xgb
            metrics_dict_xgb.setdefault("rec", {})[AU] = rec_xgb
            metrics_dict_xgb.setdefault("fscore", {})[AU] = fscore_xgb
            metrics_dict_xgb.setdefault("acc", {})[AU] = acc_xgb

        for metric in metrics_dict_svm:
            metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics_dict_svm[metric], ignore_index=True)
        for metric in metrics_dict_xgb:
            metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics_dict_xgb[metric], ignore_index=True)

  0%|                                                                                           | 0/16 [00:00<?, ?it/s]C:\Users\Daniel\AppData\Roaming\Python\Python39\site-packages\xgboost\core.py:160: UserWarning: [17:29:50] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
100%|██████████████████████████████████████████████████████████████████████████████████| 16/16 [00:25<00:00,  1.62s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarn

100%|██████████████████████████████████████████████████████████████████████████████████| 16/16 [00:17<00:00,  1.08s/it]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics

100%|██████████████████████████████████████████████████████████████████████████████████| 16/16 [00:13<00:00,  1.17it/s]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics

100%|██████████████████████████████████████████████████████████████████████████████████| 16/16 [00:15<00:00,  1.05it/s]
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics_dict_svm[metric], ignore_index=True)
C:\Users\Daniel\AppData\Local\Temp\ipykernel_22960\1031838758.py:81: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  metrics_df_anno[metric] = metrics_df_anno[metric].append(metrics

In [47]:
metrics_df_anno["fscore"]["Average"] = metrics_df_anno["fscore"].loc[:,AU_list].mean(axis=1)

In [58]:
metrics_df_anno["fscore"][metrics_df_anno["fscore"]["model"].str.contains("_0.70")].to_csv("HOG_AU_anno_eval_{}_0.7.csv".format(metric))

In [57]:
for metric in metrics_df:
    metrics_df_anno[metric].to_csv("HOG_AU_anno_eval_{}.csv".format(metric))